# STAT 8017B Project 4 — Chatbot Integration
## Financial Analysis Chatbot (Group 4.1)

This notebook integrates all trained models into a chatbot pipeline:

1. **Intent classification** — TF-IDF + cosine similarity to detect user query type
2. **Routing logic** — dispatch to the correct analysis module
3. **Knowledge base retrieval** — Q&A lookup via cosine similarity
4. **LLM response generation** — Phi-3.5 via Ollama wraps structured results into natural language
5. **End-to-end test** — sample queries

**Prerequisite**: Ollama must be installed and `phi3.5` model pulled.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

warnings.filterwarnings('ignore')
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
MODEL_DIR = os.path.join('..', 'models')

print('Setup complete.')

In [ ]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    tokens = nltk.word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

---
## 1. Load All Pre-trained Models and Data

In [ ]:
# --- Data ---
etf = pd.read_csv(os.path.join(PROCESSED_DIR, 'etf_clean.csv'))
mf = pd.read_csv(os.path.join(PROCESSED_DIR, 'mutualfund_clean.csv'))
comp = pd.read_csv(os.path.join(PROCESSED_DIR, 'complaints_clean.csv'))
qa = pd.read_csv(os.path.join(PROCESSED_DIR, 'qa_clean.csv'))
prices = pd.read_csv(os.path.join(PROCESSED_DIR, 'prices_clean.csv'))
news = pd.read_csv(os.path.join(PROCESSED_DIR, 'news_clean.csv'))

# --- Q&A retrieval ---
qa_tfidf = joblib.load(os.path.join(PROCESSED_DIR, 'qa_tfidf_vectorizer.joblib'))
qa_tfidf_matrix = joblib.load(os.path.join(PROCESSED_DIR, 'qa_tfidf_matrix.joblib'))

# --- Classification models ---
complaint_svm = joblib.load(os.path.join(MODEL_DIR, 'complaint_svm.joblib'))
complaint_tfidf = joblib.load(os.path.join(PROCESSED_DIR, 'complaints_tfidf.joblib'))
complaint_labels = joblib.load(os.path.join(PROCESSED_DIR, 'complaints_labels.joblib'))

sentiment_svm = joblib.load(os.path.join(MODEL_DIR, 'sentiment_svm.joblib'))
sentiment_tfidf = joblib.load(os.path.join(MODEL_DIR, 'sentiment_tfidf.joblib'))
sentiment_le = joblib.load(os.path.join(PROCESSED_DIR, 'phrasebank_label_encoder.joblib'))

# --- Regression models ---
fund_lr = joblib.load(os.path.join(MODEL_DIR, 'fund_return_lr.joblib'))
fund_rf = joblib.load(os.path.join(MODEL_DIR, 'fund_return_rf.joblib'))
fund_scaler = joblib.load(os.path.join(MODEL_DIR, 'fund_return_scaler.joblib'))

print('All models and data loaded.')

---
## 2. Intent Classifier

Classify user queries into intents using TF-IDF + cosine similarity against seed examples.

In [ ]:
INTENT_EXAMPLES = {
    'fund_lookup': [
        'What are the top performing ETFs',
        'Show me the best mutual funds',
        'Which ETF has the highest return',
        'Compare ETFs by expense ratio',
        'Find funds with low expense ratio and high returns',
        'What is the best tech ETF',
        'List ETFs in the large blend category',
    ],
    'complaint_analysis': [
        'What do customers complain about most',
        'Show complaint categories',
        'Which products have the most complaints',
        'Classify this complaint',
        'What are the top issues in banking complaints',
        'Analyze customer complaints',
    ],
    'sentiment': [
        'Analyze sentiment of this text',
        'Is this news positive or negative',
        'What is the sentiment of the market today',
        'Sentiment analysis',
        'Is this financial statement bullish or bearish',
    ],
    'prediction': [
        'Predict return for an ETF',
        'What will the stock price be',
        'Forecast fund performance',
        'Predict returns given expense ratio',
        'Estimate future return of tech fund',
    ],
    'price_trend': [
        'Show stock price trend',
        'What is AAPL price history',
        'Plot stock price chart',
        'How has the market performed this year',
        'Show me price data for Tesla',
    ],
    'general_qa': [
        'What is an ETF',
        'How does a mutual fund work',
        'What is expense ratio',
        'Explain P/E ratio',
        'What is revenue',
        'Define market capitalization',
    ],
}

intent_texts = []
intent_labels = []
for intent, examples in INTENT_EXAMPLES.items():
    for ex in examples:
        intent_texts.append(clean_text(ex))
        intent_labels.append(intent)

intent_tfidf = TfidfVectorizer(max_features=1000)
intent_matrix = intent_tfidf.fit_transform(intent_texts)


def classify_intent(query, threshold=0.15):
    """Return the best matching intent for a user query."""
    vec = intent_tfidf.transform([clean_text(query)])
    sims = cosine_similarity(vec, intent_matrix).flatten()
    best_idx = sims.argmax()
    best_score = sims[best_idx]
    if best_score < threshold:
        return 'general_qa', best_score
    return intent_labels[best_idx], best_score


test_queries = [
    'What are the top ETFs?',
    'Classify this complaint about credit cards',
    'Is the market sentiment positive today?',
    'Predict return for a bond fund',
    'Show AAPL price trend',
    'What is a stock split?',
]
for q in test_queries:
    intent, score = classify_intent(q)
    print(f'  "{q}"  =>  {intent} (score={score:.3f})')

---
## 3. Analysis Modules

Each module produces structured results that can be wrapped by the LLM.

In [ ]:
def module_fund_lookup(query):
    """Return top funds by 1-year return."""
    etf_sorted = etf.sort_values('fund_return_1year', ascending=False)
    top = etf_sorted[['fund_symbol', 'fund_short_name', 'fund_category',
                       'fund_return_1year', 'fund_annual_report_net_expense_ratio']].head(10)
    return {
        'type': 'fund_lookup',
        'summary': f'Top 10 ETFs by 1-year return',
        'data': top.to_dict('records')
    }


def module_complaint_analysis(query):
    """Analyze complaint categories or classify a complaint."""
    dist = comp['Product'].value_counts().head(10)
    return {
        'type': 'complaint_analysis',
        'summary': 'Top 10 complaint product categories',
        'data': dist.to_dict()
    }


def module_sentiment(query):
    """Analyze sentiment of the query text."""
    cleaned = clean_text(query)
    vec = sentiment_tfidf.transform([cleaned])
    pred = sentiment_svm.predict(vec)[0]
    label = sentiment_le.inverse_transform([pred])[0]
    scores = sentiment_svm.decision_function(vec)[0]
    return {
        'type': 'sentiment',
        'summary': f'Sentiment: {label}',
        'label': label,
        'scores': {sentiment_le.inverse_transform([i])[0]: round(float(s), 4)
                   for i, s in enumerate(scores)}
    }


def module_prediction(query):
    """Predict fund return using median feature values as default."""
    feature_cols = ['fund_annual_report_net_expense_ratio', 'total_net_assets', 'fund_yield',
                    'fund_return_ytd', 'fund_return_3years', 'fund_return_5years',
                    'fund_mean_annual_return_3years', 'fund_stdev_3years',
                    'fund_sharpe_ratio_3years', 'fund_beta_3years',
                    'asset_stocks', 'asset_bonds']
    available = [c for c in feature_cols if c in etf.columns]
    median_vals = etf[available].median().values.reshape(1, -1)
    scaled = fund_scaler.transform(median_vals)
    pred_lr = fund_lr.predict(scaled)[0]
    pred_rf = fund_rf.predict(scaled)[0]
    return {
        'type': 'prediction',
        'summary': f'Predicted 1-year return (median fund profile)',
        'linear_regression': round(float(pred_lr), 4),
        'random_forest': round(float(pred_rf), 4)
    }


def module_price_trend(query):
    """Return recent price data for a ticker."""
    available_tickers = prices['ticker'].unique()
    # Try to extract a ticker from the query
    query_upper = query.upper()
    found_ticker = None
    for t in available_tickers:
        if t.upper() in query_upper:
            found_ticker = t
            break
    if found_ticker is None:
        found_ticker = available_tickers[0]

    ticker_data = prices[prices['ticker'] == found_ticker].tail(30)
    price_col = 'Close' if 'Close' in ticker_data.columns else 'Adj Close'
    return {
        'type': 'price_trend',
        'summary': f'Last 30 days of {found_ticker}',
        'ticker': found_ticker,
        'data': ticker_data[['Date', price_col, 'daily_return', 'ma_20', 'ma_50']].to_dict('records')
    }


def module_general_qa(query):
    """Retrieve best matching Q&A pair from knowledge base."""
    vec = qa_tfidf.transform([clean_text(query)])
    sims = cosine_similarity(vec, qa_tfidf_matrix).flatten()
    top_idx = sims.argsort()[-3:][::-1]
    results = []
    for idx in top_idx:
        results.append({
            'score': round(float(sims[idx]), 4),
            'question': qa.iloc[idx]['question'],
            'answer': qa.iloc[idx]['answer']
        })
    return {
        'type': 'general_qa',
        'summary': 'Retrieved from Financial Q&A knowledge base',
        'results': results
    }


MODULE_MAP = {
    'fund_lookup': module_fund_lookup,
    'complaint_analysis': module_complaint_analysis,
    'sentiment': module_sentiment,
    'prediction': module_prediction,
    'price_trend': module_price_trend,
    'general_qa': module_general_qa,
}

print('All analysis modules defined.')

---
## 4. LLM Integration (Phi-3.5 via Ollama)

Feed structured results into Phi-3.5 to generate a natural language response.

In [ ]:
import json

USE_LLM = True

try:
    from ollama import chat
    # Quick connectivity test
    _ = chat(model='phi3.5', messages=[{'role': 'user', 'content': 'Hi'}])
    print('Ollama + Phi-3.5 connected successfully.')
except Exception as e:
    print(f'Ollama not available: {e}')
    print('Falling back to template-based responses.')
    USE_LLM = False

In [ ]:
SYSTEM_PROMPT = """You are a Financial Analysis Chatbot. You help users understand financial products,
analyze complaints, perform sentiment analysis, and predict fund performance.
You are given structured analysis results from our data pipeline.
Summarize the results in clear, concise natural language for a retail investor.
If data is provided, reference specific numbers. Keep answers under 200 words."""


def generate_response(query, analysis_result):
    """Generate natural language response using LLM or template fallback."""
    context = json.dumps(analysis_result, indent=2, default=str)

    if USE_LLM:
        try:
            response = chat(
                model='phi3.5',
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': f'User question: {query}\n\nAnalysis results:\n{context}'}
                ]
            )
            return response['message']['content']
        except Exception as e:
            pass

    # Template fallback
    return f"**{analysis_result.get('summary', 'Analysis complete')}**\n\n```json\n{context}\n```"


print('Response generator ready.')

---
## 5. Full Chatbot Pipeline

In [ ]:
def chatbot(query):
    """Full chatbot pipeline: classify intent -> run module -> generate response."""
    print(f'\nUser: {query}')
    print('-' * 50)

    intent, score = classify_intent(query)
    print(f'Intent: {intent} (confidence: {score:.3f})')

    module_fn = MODULE_MAP.get(intent, module_general_qa)
    result = module_fn(query)

    response = generate_response(query, result)
    print(f'\nBot: {response}')
    print('=' * 50)
    return response

print('Chatbot pipeline ready.')

---
## 6. End-to-End Test

Test with sample queries from the project plan.

In [ ]:
test_queries = [
    'What are the top performing ETFs?',
    'What do customers complain about most?',
    'Analyze sentiment: The market showed strong gains today in technology sector.',
    'Predict return for a tech ETF with 0.5% expense ratio',
    'Show AAPL price trend',
    'What is an ETF?',
]

for q in test_queries:
    chatbot(q)

---
## 7. Save Chatbot Components

Save the intent classifier and module map for the Streamlit app.

In [ ]:
joblib.dump(intent_tfidf, os.path.join(MODEL_DIR, 'intent_tfidf.joblib'))
joblib.dump(intent_matrix, os.path.join(MODEL_DIR, 'intent_matrix.joblib'))
joblib.dump(intent_labels, os.path.join(MODEL_DIR, 'intent_labels.joblib'))

print('Chatbot components saved to models/ directory.')
for f in sorted(os.listdir(MODEL_DIR)):
    size_kb = os.path.getsize(os.path.join(MODEL_DIR, f)) / 1024
    print(f'  {f:40s} {size_kb:8.1f} KB')